# OCR‑1 Inference with CRNN + LLM Post‑Processing

This notebook demonstrates how to load a trained CRNN model and run inference on historical document images.
It also shows optional LLM correction using Gemini.

**Author:** Abhiram G (GSoC 2026 applicant)
**Repository:** humanai-foundation/RenAIssance

## 1. Setup
Install required packages (if not already installed).

In [ ]:
!pip install torch torchvision matplotlib pillow opencv-python
!pip install google-generativeai  # optional, for LLM post‑processing

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import sys
import os

# Add paths to the repository
sys.path.append('../RenAIssance_SelfSupervisedLearning_OCR_YukinoriYamamoto')
from ResNet import ResNet18, ResNet34

# Import LLM module if available
try:
    sys.path.append('..')
    from llm_postprocess import postprocess_ocr
    llm_available = True
except ImportError:
    llm_available = False
    print("LLM module not found. Post‑processing will be skipped.")

## 2. Load Pretrained Model
Replace `model_path` with the actual path to your trained weights.

If you don't have a trained model, we provide a dummy placeholder to demonstrate the pipeline.

In [ ]:
# Model configuration
num_classes = 80  # character set size (adjust if needed)
model = ResNet18(num_classes=num_classes)

# Load weights (update path)
model_path = "path/to/your/checkpoint.pth"  # TODO: change
if os.path.exists(model_path):
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    print("Model loaded successfully.")
else:
    print(f"Warning: Model file not found at {model_path}. Using untrained model.")

model.eval()

## 3. Image Preprocessing
Convert input image to the format expected by the CRNN.

In [ ]:
def preprocess_image(image_path, target_height=32):
    """Load and preprocess an image for CRNN input."""
    img = Image.open(image_path).convert('L')  # grayscale
    # Resize to fixed height, keep aspect ratio for width
    w, h = img.size
    new_width = int(w * target_height / h)
    img = img.resize((new_width, target_height), Image.BILINEAR)
    
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])
    tensor = transform(img).unsqueeze(0)  # add batch dimension
    return tensor, img.size

## 4. Decoding Predictions
Convert model output (probability distribution over time steps) into text using greedy decoding.
This uses a simple character mapping; replace with your actual character set.

In [ ]:
# Example character set (update to match your training)
char_set = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 .,;:?!-'",
char_to_idx = {c: i for i, c in enumerate(char_set)}
idx_to_char = {i: c for c, i in char_to_idx.items()}

def greedy_decode(output):
    """Greedy decoding from model output (B, T, C)."""
    probs = torch.softmax(output, dim=-1)
    pred_indices = torch.argmax(probs, dim=-1).squeeze(0).tolist()
    # collapse repeated and remove blanks (if blank index known)
    # For simplicity, we convert each index to char
    text = ''.join([idx_to_char.get(idx, '?') for idx in pred_indices])
    return text

## 5. Run Inference
Test on a sample image. Replace `sample_image_path` with an actual image.

In [ ]:
sample_image_path = "sample_historical_document.png"  # TODO: change

if not os.path.exists(sample_image_path):
    print(f"Sample image not found at {sample_image_path}. Please provide a valid image.")
else:
    # Preprocess
    input_tensor, original_size = preprocess_image(sample_image_path)
    
    # Inference
    with torch.no_grad():
        output = model(input_tensor)  # shape: (1, T, num_classes)
    
    # Decode
    raw_text = greedy_decode(output)
    print("Raw OCR output:", raw_text)
    
    # Optional LLM correction
    if llm_available and 'GEMINI_API_KEY' in os.environ:
        corrected = postprocess_ocr(raw_text, api_key=os.environ['GEMINI_API_KEY'])
        print("LLM corrected:", corrected['corrected_text'])
        print("CER improvement:", corrected['cer_improvement'])
    
    # Display image
    img = Image.open(sample_image_path)
    plt.imshow(img, cmap='gray')
    plt.axis('off')
    plt.show()